# Lab 4 — From pixels to a robot motion: calibration and guidance of the arm

**Deep Learning for Computer Vision — Day 1** · ECAM, 5MAU · 2026–2027

This lab closes the day: **a camera guides the robot arm of the project** to a red circle placed on the table. At the end you will have:

* the intrinsics $K$ and the distortion coefficients of a camera, with a measure of their quality;
* a function converting a detected part `(u, v)` into `(X, Y)` **in millimetres** in the table frame, and a quantitative answer to *"does calibration matter?"*;
* the transform from the table to the **robot frame**, and a safe motion sequence sent to the arm;
* a `calibration.yaml` file you can reuse in the project.

**Organisation (~90 min).**
1. **Everybody**, on the **common dataset** (`data/calib`, `data/table`, `data/robot`): same images for all, so we can compare results. Priority: sections 1–4, 6.1–6.2 and 9. Sections 5, 6.3, 7 and 8 can be finished at home.
2. **In groups, at the robot cell** (section 10): calibrate the cell camera, teach the table to the arm and send it to the red circle. Until it is your turn at the robot, everything runs in simulation.

**About the common dataset.** The images were rendered with a *simulated* wide-angle USB camera whose true parameters are stored in `synthetic_ground_truth.json` — you can check your results (section 8). The robot is simulated too (`SimulatedRobot`, section 9), with the same commands as the real arm.

> *References:* Z. Zhang, "A flexible new technique for camera calibration", *IEEE TPAMI* 22(11), 2000 · OpenCV tutorials *Camera calibration* and *Detection of ArUco markers*, https://docs.opencv.org/ · R. Hartley & A. Zisserman, *Multiple View Geometry in Computer Vision*, 2nd ed., 2004 · Q. Lurkin, *Image Processing – Lab 1: Camera Geometry*, ECAM · Galatae, *galatae-api* (Python API of the Gal-e arm, GPL-3.0), https://github.com/galatae-engineering/galatae-api

In [ ]:
# Setup check: are all the packages of requirements.txt installed in this kernel?
import importlib
import importlib.util

importlib.invalidate_caches()
REQUIRED = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib", "cv2": "opencv-contrib-python",
            "ipywidgets": "ipywidgets", "ipympl": "ipympl", "serial": "pyserial"}
missing = [pkg for module, pkg in REQUIRED.items() if importlib.util.find_spec(module) is None]
if "opencv-contrib-python" not in missing:
    import cv2
    if not hasattr(cv2, "aruco"):  # opencv-python (without contrib) shadows opencv-contrib-python
        missing.append("opencv-contrib-python (run `%pip uninstall -y opencv-python opencv-contrib-python` "
                       "then `%pip install \"opencv-contrib-python<5\"`)")
assert not missing, (f"Missing packages: {missing}. Run `%pip install -r ../requirements.txt` "
                     "in a cell, then restart the kernel (Kernel > Restart) and run the notebook again.")
print("Setup OK: all required packages are installed.")

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

DATA = Path("data") if Path("data").exists() else Path("../data")
np.set_printoptions(precision=4, suppress=True)
calib_files = sorted((DATA / "calib").glob("*.png"))
print(len(calib_files), "calibration images")

fig, ax = plt.subplots(2, 4, figsize=(16, 5))
for a, f in zip(ax.ravel(), calib_files[::3]):
    a.imshow(cv2.imread(str(f))[:, :, ::-1]); a.set_title(f.name); a.axis("off")

## 1. Detect the checkerboard corners

The board has **9 × 6 inner corners** and squares of **25 mm**. For each image:
1. `cv2.findChessboardCorners(gray, (9, 6))` finds the corners roughly (or fails);
2. `cv2.cornerSubPix` refines them to sub-pixel accuracy (essential for a good calibration).

### Exercise 1.1
Fill the lists `image_points` (refined corners) and `used_files`. Print the files where the detection fails. Why does it fail?

In [ ]:
PATTERN = (9, 6)
SQUARE_MM = 25.0
criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 1e-3)

image_points, used_files = [], []
for f in calib_files:
    gray = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
    # TODO
    ...
image_size = gray.shape[::-1]  # (width, height)
print(len(image_points), "images usable, image size", image_size)

vis = cv2.imread(str(used_files[5]))
cv2.drawChessboardCorners(vis, PATTERN, image_points[5], True)
plt.figure(figsize=(9, 5)); plt.imshow(vis[:, :, ::-1]); plt.axis("off");

## 2. The 3D points of the board

Zhang's method uses the board as the world frame: the board lies in the plane $Z = 0$ and the corner $(i, j)$ is at $(i \cdot 25,\ j \cdot 25,\ 0)$ mm, in the same order as `findChessboardCorners` returns them (row by row).

### Exercise 2.1
Build `objp`, an array of shape `(54, 3)`, `float32`.

In [ ]:
# TODO
objp = ...
assert objp.shape == (54, 3) and np.allclose(objp[1], [25, 0, 0]) and np.allclose(objp[9], [0, 25, 0])
object_points = [objp] * len(image_points)

## 3. Calibrate

`cv2.calibrateCamera` estimates $K$, the distortion $(k_1, k_2, p_1, p_2, k_3)$ and one pose $(R_i, t_i)$ per image by minimising the **reprojection error**: the distance between the detected corners and the corners projected with the estimated model. It returns the RMS of this error, in pixels.

In [ ]:
rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(object_points, image_points, image_size, None, None)
print(f"RMS reprojection error: {rms:.3f} px")
print("K =\n", K)
print("dist (k1, k2, p1, p2, k3) =", dist.ravel())

## 4. Per-image reprojection error

A global RMS hides bad images. A single bad image (motion blur, wrong detection) can spoil the calibration.

### Exercise 4.1
For each image, project `objp` with `cv2.projectPoints` and the estimated pose, and compute the RMS distance to the detected corners. Plot a bar chart. Which image is an outlier, and why?

In [ ]:
def per_image_errors(object_points, image_points, rvecs, tvecs, K, dist):
    errors = []
    # TODO
    ...
    return np.array(errors)


errors = per_image_errors(object_points, image_points, rvecs, tvecs, K, dist)
plt.figure(figsize=(12, 3))
plt.bar([f.stem[-2:] for f in used_files], errors)
plt.axhline(rms, color="r", ls="--", label="global RMS"); plt.ylabel("RMS error [px]"); plt.legend();
worst = used_files[int(np.argmax(errors))]
print("worst image:", worst.name, f"({errors.max():.2f} px)")

In [ ]:
plt.figure(figsize=(9, 5)); plt.imshow(cv2.imread(str(worst))[:, :, ::-1]); plt.axis("off");

### Exercise 4.2
Remove the outlier(s) (error > 1 px) and calibrate again. Compare the RMS and the parameters.

In [ ]:
# TODO
...
rms, K, dist, rvecs, tvecs = cv2.calibrateCamera(object_points, image_points, image_size, None, None)
print(f"{len(image_points)} images | RMS = {rms:.3f} px")
print("K =\n", K, "\ndist =", dist.ravel())
assert rms < 0.5

## 5. Undistort

`cv2.getOptimalNewCameraMatrix(K, dist, size, alpha)` chooses the camera matrix of the corrected image: with `alpha=0` only valid pixels are kept (the image is cropped/zoomed), with `alpha=1` all source pixels are kept (black borders).

### Exercise 5.1
Undistort `table_normal.png` and one checkerboard image with `alpha = 0` and `alpha = 1`. Check that the edges of the board become straight.

In [ ]:
table = cv2.imread(str(DATA / "table" / "table_normal.png"))
board = cv2.imread(str(used_files[3]))
fig, ax = plt.subplots(2, 3, figsize=(18, 7))
for row, im in enumerate([board, table]):
    ax[row, 0].imshow(im[:, :, ::-1]); ax[row, 0].set_title("original")
    for col, alpha in enumerate([0, 1], start=1):
        # TODO
        und = ...
        ax[row, col].imshow(und[:, :, ::-1]); ax[row, col].set_title(f"undistorted, alpha={alpha}")
for a in ax.ravel():
    a.axis("off")

## 6. From pixels to millimetres on the table

The table carries 4 **ArUco markers** (dictionary `DICT_4X4_50`, ids 0–3, 40 mm) whose centres are at known positions in the table frame (mm). They define the frame of the workspace — the same frame you will express in the robot frame in the project.

| id | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| (X, Y) mm | (0, 0) | (400, 0) | (400, 280) | (0, 280) |

### Exercise 6.1 — Detect the markers
1. Create a `cv2.aruco.ArucoDetector` for the dictionary above and call `detectMarkers` on `table`. It returns `corners` (one 1 × 4 × 2 array per marker), `ids` (N × 1) and the rejected candidates. Keep the names `corners` and `ids`: the display below uses them.
2. Build the dictionary `marker_corners`: marker id → 4 × 2 array of its corners in pixels. The centre of each marker (mean of its 4 corners) is then computed for you.

In [ ]:
MARKERS_MM = {0: (0.0, 0.0), 1: (400.0, 0.0), 2: (400.0, 280.0), 3: (0.0, 280.0)}
MARKER_SIZE = 40.0
table = cv2.imread(str(DATA / "table" / "table_normal.png"))

# TODO
corners, ids, rejected = ...  # detect the markers in table
# TODO
marker_corners = ...  # dictionary {id: 4 x 2 array of corners in pixels}
marker_centres = {i: c.mean(axis=0) for i, c in marker_corners.items()}
assert sorted(marker_centres) == [0, 1, 2, 3]
vis = table.copy()
cv2.aruco.drawDetectedMarkers(vis, corners, ids)
plt.figure(figsize=(9, 5)); plt.imshow(vis[:, :, ::-1]); plt.axis("off");

### Exercise 6.2 — Does calibration matter?
A simple part detector (colour threshold + moments, see Lecture 2) is given below. Convert the detected centres to millimetres with a homography estimated from the 4 marker centres, in two ways:

* **A — no calibration:** homography between the *raw* pixel coordinates and the table;
* **B — with calibration:** first undistort the pixel coordinates (`cv2.undistortPoints(..., P=K)`), for the markers *and* the parts.

Compare both with the true positions of the parts (measured on the table) and fill in the table of errors.

In [ ]:
def detect_parts(img_bgr, min_area=300):
    """Colour threshold + moments: list of dicts with the centre (u, v) and colour of each coloured part."""
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 90, 40), (179, 255, 255))
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    parts = []
    for c in contours:
        m = cv2.moments(c)
        if m["m00"] >= min_area:
            parts.append({"u": m["m10"] / m["m00"], "v": m["m01"] / m["m00"]})
    return parts


truth = json.loads((DATA / "synthetic_ground_truth.json").read_text())["table"]["parts"]
true_xy = np.array([[p["x_mm"], p["y_mm"]] for p in truth])
names = [p["name"] for p in truth]
parts = detect_parts(table)
uv_parts = np.array([[p["u"], p["v"]] for p in parts], np.float32)
uv_markers = np.array([marker_centres[i] for i in range(4)], np.float32)
xy_markers = np.array([MARKERS_MM[i] for i in range(4)], np.float32)


def undistort_px(uv, K, dist):
    return cv2.undistortPoints(uv.reshape(-1, 1, 2).astype(np.float32), K, dist, P=K).reshape(-1, 2)


def match_to_truth(est_xy):
    """Associate each true part with the closest estimate; return the errors in mm."""
    return np.array([np.min(np.linalg.norm(est_xy - t, axis=1)) for t in true_xy])


# TODO
xy_A = ... ; xy_B = ...   (N x 2 arrays in mm)
err_A, err_B = match_to_truth(xy_A), match_to_truth(xy_B)
print(f"{'part':15s} {'A: raw [mm]':>12s} {'B: calibrated [mm]':>20s}")
for n, a, b in zip(names, err_A, err_B):
    print(f"{n:15s} {a:12.2f} {b:20.2f}")
print(f"{'mean':15s} {err_A.mean():12.2f} {err_B.mean():20.2f}")
assert err_B.mean() < 1.0

### Exercise 6.3 — Camera pose and parts with a height
A homography is only valid **on the table plane**. For a part of height $h$ (e.g. a 30 mm cube), we need the full 3D model.

1. Use `cv2.solvePnP` with the 16 marker corners (3D in the table frame, $Z = 0$) to estimate the pose $(R, t)$ of the camera w.r.t. the table. The ArUco corners are returned in the order top-left, top-right, bottom-right, bottom-left of the marker.
2. Compute the position of the camera centre $C = -R^T t$ in the table frame. *Note:* the table frame has X to the right, Y "down" in the image, so Z (= X × Y) points **into** the table: the camera is at negative Z.
3. Write `pixel_to_plane(uv, K, dist, R, t, z)` intersecting the ray of a pixel with the plane $Z = z$. For the top face of a part of height $h$, use $z = -h$.

In [ ]:
# TODO
...
print("camera centre in the table frame [mm]:", C)
print(f"height of the camera above the table: {-C[2]:.1f} mm")


def pixel_to_plane(uv, K, dist, R, t, z=0.0):
    """uv: N x 2 pixels -> N x 2 points (X, Y) on the plane Z = z of the table frame."""
    # TODO
    ...


xy_C = pixel_to_plane(uv_parts, K, dist, R_tc, tvec, z=0.0)
print("mean error with the 3D model (flat parts):", match_to_truth(xy_C).mean().round(2), "mm")

**Parallax.** A cube of height 30 mm stands at $(X, Y) = (350, 250)$ mm. Its top face centre is seen at the pixel computed below. Where does the *table homography* place it? And `pixel_to_plane` with $z = -30$?

In [ ]:
top = np.array([[350.0, 250.0, -30.0]])
uv_top, _ = cv2.projectPoints(top, rvec, tvec, K, dist)
uv_top = uv_top.reshape(1, 2)
xy_homography = cv2.perspectiveTransform(undistort_px(uv_top, K, dist).reshape(-1, 1, 2), H_B).ravel()
xy_3d = pixel_to_plane(uv_top, K, dist, R_tc, tvec, z=-30.0).ravel()
print("homography (assumes z = 0):", xy_homography.round(1), "-> error",
      np.linalg.norm(xy_homography - [350, 250]).round(1), "mm")
print("3D model with z = -30     :", xy_3d.round(1))

## 7. Save the calibration

Save $K$, the distortion, the image size and the RMS in a YAML file with `cv2.FileStorage`, and check you can read it back. You will do the same with the project camera.

In [ ]:
fs = cv2.FileStorage("calibration.yaml", cv2.FILE_STORAGE_WRITE)
fs.write("image_width", image_size[0]); fs.write("image_height", image_size[1])
fs.write("K", K); fs.write("dist", dist); fs.write("rms", rms)
fs.release()

fs = cv2.FileStorage("calibration.yaml", cv2.FILE_STORAGE_READ)
assert np.allclose(fs.getNode("K").mat(), K)
fs.release()
print(Path("calibration.yaml").read_text()[:400])

## 8. Check against the truth (common dataset only)

Because the common dataset is synthetic, we know the true parameters. How close are you?

In [ ]:
gt = json.loads((DATA / "synthetic_ground_truth.json").read_text())
K_true, dist_true = np.array(gt["K"]), np.array(gt["dist_k1_k2_p1_p2_k3"])
print("            estimated      true")
for name, a, b in [("fx", K[0, 0], K_true[0, 0]), ("fy", K[1, 1], K_true[1, 1]),
                   ("cx", K[0, 2], K_true[0, 2]), ("cy", K[1, 2], K_true[1, 2])]:
    print(f"{name:6s} {a:12.1f} {b:10.1f}")
for name, a, b in zip(["k1", "k2", "p1", "p2", "k3"], dist.ravel(), dist_true):
    print(f"{name:6s} {a:12.4f} {b:10.4f}")

**Question.** On this clean synthetic dataset the estimate is very close to the truth. With real images, $k_2$ and $k_3$ often change a lot between two calibrations of the same camera while the reprojection error stays tiny. Why is it not a problem in practice?

## 9. From the camera to the robot arm

The robot of the project is a **Galatae Gal-e** arm (5 axes + gripper, reach 60 cm), driven from Python over USB with the `galatae-api` library. A pose is a list `[x, y, z, pitch, roll]` in mm and degrees, in the **robot frame** (origin at the base, $z$ up); `pitch = 180` points the tool straight down.

**The task.** A red circle (diameter 30 mm) lies on the table among the 4 markers, with other parts around it. Find it in the image and bring the tool **20 mm above its centre**. Everything you need was built today:

$$(u, v) \xrightarrow{\ \text{undistort}\ +\ H\ } (X, Y)_{table} \xrightarrow{\ \text{table} \to \text{robot}\ } (x, y, z)_{robot} \xrightarrow{\ \text{safe moves}\ } \text{robot}$$

We first work in simulation with the common dataset (`data/robot/robot_cell_*.png`, same camera and table as above). Section 10 does the same with the real camera and the real arm.

### Exercise 9.1 — Teach the table to the robot
Where is the table in the robot frame? The operator moves the tool tip onto the centre of each marker and reads the pose with `robot.get_tool_pose()`. The four readings are in `data/robot/taught_markers.json` (in section 10 you measure them yourself).

The table is a plane, so the change of frame is a 2D **affine** map plus a constant height $z_{table}$:
$$x_r = a\,X + b\,Y + c, \qquad y_r = d\,X + e\,Y + f$$
Write `fit_table_to_robot` estimating the $2 \times 3$ matrix $A = \begin{pmatrix} a & b & c \\ d & e & f \end{pmatrix}$ by least squares (`np.linalg.lstsq`) from the 4 pairs of points: 8 equations, 6 unknowns. Look at the residuals.

In [ ]:
taught = json.loads((DATA / "robot" / "taught_markers.json").read_text())["markers"]
xy_table = np.array([MARKERS_MM[i] for i in range(4)])
xyz_taught = np.array([taught[str(i)] for i in range(4)])
print("taught tool positions [mm, robot frame]:\n", xyz_taught)


def fit_table_to_robot(xy_table, xy_robot):
    """N x 2 table points and N x 2 robot points -> 2 x 3 affine matrix A (least squares)."""
    # TODO
    ...


def apply_affine(A, xy):
    xy = np.atleast_2d(xy)
    return xy @ A[:, :2].T + A[:, 2]


A_tr = fit_table_to_robot(xy_table, xyz_taught[:, :2])
z_table = xyz_taught[:, 2].mean()
residuals = np.linalg.norm(apply_affine(A_tr, xy_table) - xyz_taught[:, :2], axis=1)
print("A =\n", A_tr.round(3))
print("residuals [mm]:", residuals.round(2), f"| table height in the robot frame: z = {z_table:.1f} mm")
assert A_tr.shape == (2, 3) and residuals.max() < 1.0

**Question.** The $2 \times 2$ part of $A$ is close to $\begin{pmatrix} 0 & 1 \\ 1 & 0 \end{pmatrix}$. What does it mean for the axes? Why could a rotation + translation not represent it? Where do the residuals come from?

### Exercise 9.2 — Find the red circle
The red mask is given (the red hue wraps around 0/180, hence two ranges). There are **two** red objects: the circle and a square. Keep the contour with the highest **circularity** $4\pi A / P^2$ ($A$ = area, $P$ = perimeter: 1 for a perfect disc, $\pi/4 \approx 0.79$ for a square) and return the centre $(u, v)$ of its moments.

In [ ]:
def red_mask(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, (0, 120, 60), (10, 255, 255)) | cv2.inRange(hsv, (170, 120, 60), (179, 255, 255))
    return cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((5, 5), np.uint8))


def find_red_circle(img_bgr, min_area=200):
    """Centre (u, v) in pixels of the most circular red blob, and its circularity."""
    contours, _ = cv2.findContours(red_mask(img_bgr), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    best_uv, best_circ = None, 0.0
    # TODO
    ...
    return best_uv, best_circ


scenes = sorted((DATA / "robot").glob("robot_cell_*.png"))
fig, ax = plt.subplots(1, len(scenes), figsize=(6 * len(scenes), 4))
for a, f in zip(ax, scenes):
    img = cv2.imread(str(f))
    uv, circ = find_red_circle(img)
    a.imshow(img[:, :, ::-1]); a.plot(*uv, "c+", ms=25, mew=3)
    a.set_title(f"{f.name}: circularity {circ:.2f}"); a.axis("off")
assert all(find_red_circle(cv2.imread(str(f)))[1] > 0.85 for f in scenes)

### Exercise 9.3 — From a pixel to the robot frame
`table_homography(img)` re-detects the markers **in each new image** (the camera may have been bumped) and returns the homography *undistorted pixels → table mm* of exercise 6.2. Complete `pixel_to_robot`, then compare with the true positions of the circles.

In [ ]:
detector = cv2.aruco.ArucoDetector(cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50))


def table_homography(img_bgr, K, dist, markers_mm=MARKERS_MM):
    corners, ids, _ = detector.detectMarkers(img_bgr)
    assert ids is not None and set(markers_mm) <= set(ids.ravel()), "all the markers must be visible"
    centres = {int(i): c.reshape(4, 2).mean(axis=0) for c, i in zip(corners, ids.ravel())}
    uv = np.array([centres[i] for i in markers_mm], np.float32)
    xy = np.array([markers_mm[i] for i in markers_mm], np.float32)
    return cv2.getPerspectiveTransform(undistort_px(uv, K, dist), xy)


def pixel_to_robot(uv, H_table, A_tr, K, dist):
    """N x 2 pixels -> N x 2 robot coordinates (x, y) in mm."""
    # TODO
    ...


gt = json.loads((DATA / "synthetic_ground_truth.json").read_text())
truth_robot = {s["file"]: s["circle_robot_mm"] for s in gt["robot"]["scenes"]}
errors = []
for f in scenes:
    img = cv2.imread(str(f))
    uv, _ = find_red_circle(img)
    target = pixel_to_robot(uv, table_homography(img, K, dist), A_tr, K, dist)[0]
    errors.append(np.linalg.norm(target - truth_robot[f.name]))
    print(f"{f.name}: pixel {uv.round(1)} -> robot (x, y) = {target.round(1)} mm | true {truth_robot[f.name]} "
          f"| error {errors[-1]:.2f} mm")
assert max(errors) < 2.0

### Exercise 9.4 — Move the robot (simulation)
Never send a pose to a robot without checking it. `SimulatedRobot` has the same methods as `Robot` in `galatae-api` (`reset`, `home`, `set_joint_speed`, `go_to_pose`, `get_tool_pose`, `go_to_sleep_pos`): it prints the G-code line that the real library would send over USB, and keeps the trajectory. `safe_move` checks every pose against the workspace before sending it.

Complete `go_above(robot, xy, z_table)`: a safe sequence of 3 poses, tool pointing down (`pitch = 180`, `roll = 0`):
1. **approach**: above the target at $z_{table} + 80$ mm;
2. **target**: descend to $z_{table} + 20$ mm (we only *point* at the circle);
3. **retreat**: back to $z_{table} + 80$ mm.

In [ ]:
REACH_MM = (120.0, 550.0)  # allowed horizontal distance from the base (stay away from the limits)


def check_pose(pose, z_table):
    x, y, z = pose[:3]
    r = np.hypot(x, y)
    assert REACH_MM[0] <= r <= REACH_MM[1], f"out of reach: r = {r:.0f} mm"
    assert z >= z_table + 10, f"too low: z = {z:.1f} mm (table at {z_table:.1f} mm)"


def safe_move(robot, pose, z_table):
    check_pose(pose, z_table)
    robot.go_to_pose([float(p) for p in pose])


class SimulatedRobot:
    """Same interface as galatae-api's Robot: prints the G-code instead of moving an arm."""

    def __init__(self):
        self.pose, self.history = [300.0, 0.0, 200.0, 180.0, 0.0], []

    def _send(self, gcode):
        print("  ->", gcode)

    def reset(self):
        self._send("$RST=*")

    def home(self):
        self._send("$H")

    def set_joint_speed(self, speed):
        self._send(f"F{speed}")

    def go_to_pose(self, pose):
        self._send("G1" + "".join(f"{w}{round(v, 2)}" for w, v in zip("XYZAB", pose)))
        self.pose = list(pose)
        self.history.append(list(pose))

    def get_tool_pose(self):
        return list(self.pose)

    def go_to_sleep_pos(self):
        self._send("G28")


def go_above(robot, xy, z_table, hover=20.0, safe=80.0):
    # TODO
    ...


robot = SimulatedRobot()
robot.set_joint_speed(20)
img = cv2.imread(str(DATA / "robot" / "robot_cell_1.png"))
uv, _ = find_red_circle(img)
target = pixel_to_robot(uv, table_homography(img, K, dist), A_tr, K, dist)[0]
print("target:", target.round(1))
go_above(robot, target, z_table)

traj = np.array(robot.history)
assert len(traj) == 3 and np.allclose(traj[1, :2], target) and np.isclose(traj[1, 2], z_table + 20)
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ax[0].plot(*xyz_taught[[0, 1, 2, 3, 0], :2].T[::-1], "k--", label="table (taught markers)")
ax[0].plot(target[1], target[0], "ro", ms=12, label="red circle")
ax[0].plot(0, 0, "bs", ms=12, label="robot base")
th = np.linspace(-np.pi / 2, np.pi / 2, 100)
ax[0].plot(REACH_MM[1] * np.sin(th), REACH_MM[1] * np.cos(th), "b:", label="reach limit")
ax[0].set(xlabel="y robot [mm]", ylabel="x robot [mm]", title="top view (robot frame)", aspect="equal")
ax[0].invert_xaxis(); ax[0].legend(loc="lower left", fontsize=8)
r_traj = np.hypot(traj[:, 0], traj[:, 1])
ax[1].plot(r_traj, traj[:, 2], "o-", label="tool tip: approach, target, retreat")
ax[1].plot([r_traj[1] - 15, r_traj[1] + 15], [z_table, z_table], "r", lw=6, label="red circle")
ax[1].axhline(z_table, color="k", lw=1); ax[1].plot(0, z_table, "bs", ms=12)
ax[1].set(xlabel="horizontal distance from the base [mm]", ylabel="z [mm]", title="side view",
          xlim=(-20, REACH_MM[1]), ylim=(z_table - 20, z_table + 150)); ax[1].legend(fontsize=8)
plt.tight_layout()

**Question.** Why approach from above instead of going straight to the target? What happens if the pitch is not 180°? What would change for a part with a height of 30 mm?

## 10. Live: the real camera and the real Galatae arm (groups)

Ask the teacher before running the robot for the first time. Keep a hand near the emergency stop, start with a **low speed**, and keep the cell clear. If you interrupt a cell (■) or if it fails, the current movement finishes and the arm goes back to its rest position, so that you can restart; to stop **at once**, use the emergency stop.

1. **Calibrate the cell camera** (fixed above the table, focus fixed): from the `notebooks` folder, run `python ../tools/capture_calibration.py --camera 1 --out cell_calib` and take **15–25 views** of a checkerboard glued on a rigid board (tilted ±30–45°, near and far, covering the **corners of the image**). Set `MY_FOLDER` below and run the cell: it calibrates and saves `cell_calibration.yaml`.
2. **Place the 4 markers** (`data/robot/robot_cell_cutouts_A4.pdf`, print at 100 %: 40 mm squares, placement plan on page 1) around the robot workspace and measure their centres roughly with a ruler (`MARKERS_LIVE`).
3. **Teach** the markers to the robot: move the tool tip onto each marker centre (touchscreen or `robot.jog`) and read the pose.
4. **Put the red circle** on the table, take a picture, and send the robot above it. After the teaching and after each move, the arm folds away (`leave_view`): check that it no longer hides the markers or the table from the camera.

In [ ]:
MY_FOLDER = None  # e.g. Path("cell_calib")
MY_PATTERN, MY_SQUARE_MM = (9, 6), 25.0

if MY_FOLDER is not None:
    my_objp = np.zeros((MY_PATTERN[0] * MY_PATTERN[1], 3), np.float32)
    my_objp[:, :2] = np.mgrid[0:MY_PATTERN[0], 0:MY_PATTERN[1]].T.reshape(-1, 2) * MY_SQUARE_MM
    pts, files = [], []
    for f in sorted(Path(MY_FOLDER).glob("*.png")):
        g = cv2.imread(str(f), cv2.IMREAD_GRAYSCALE)
        found, c = cv2.findChessboardCorners(g, MY_PATTERN)
        if found:
            pts.append(cv2.cornerSubPix(g, c, (11, 11), (-1, -1), criteria)); files.append(f.name)
    my_rms, my_K, my_dist, my_r, my_t = cv2.calibrateCamera([my_objp] * len(pts), pts, g.shape[::-1], None, None)
    print(f"{len(pts)} views, RMS = {my_rms:.3f} px\nK =\n{my_K}\ndist = {my_dist.ravel()}")
    e = per_image_errors([my_objp] * len(pts), pts, my_r, my_t, my_K, my_dist)
    plt.figure(figsize=(12, 3)); plt.bar(files, e); plt.xticks(rotation=90);
    fs = cv2.FileStorage("cell_calibration.yaml", cv2.FILE_STORAGE_WRITE)
    fs.write("K", my_K); fs.write("dist", my_dist); fs.write("rms", my_rms); fs.release()

In [ ]:
import contextlib

LIVE_CAMERA = False  # True: take the picture with the USB camera of the cell
LIVE_ROBOT = False   # True: send the commands to the Galatae arm (ask the teacher first!)
CAMERA_ID = 1        # 0 is usually the laptop webcam
ROBOT_PORT = None    # e.g. "COM5" if the arm is not found automatically
MARKERS_LIVE = {0: (0.0, 0.0), 1: (400.0, 0.0), 2: (400.0, 280.0), 3: (0.0, 280.0)}  # measured centres, mm

if LIVE_ROBOT:
    import urllib.request

    if not Path("robot.py").exists():  # official Python API of the arm (GPL-3.0), one file
        urllib.request.urlretrieve(
            "https://raw.githubusercontent.com/galatae-engineering/galatae-api/main/robot.py", "robot.py")
    import sys

    import serial
    from robot import Robot

    sys.path.append(next((str(p) for p in (Path("../tools"), Path("../../tools")) if (p / "safe_robot.py").exists()), "."))
    from safe_robot import SafeRobot  # interrupted cell, error, kernel stopped -> the arm goes back to rest

    galatae = Robot()  # looks for the controller (Arduino "Giga") on the USB ports
    if ROBOT_PORT is not None:
        galatae.arduino = serial.Serial(port=ROBOT_PORT, baudrate=115200, timeout=0.1)
    arm = SafeRobot(galatae)
    with arm.guard():
        arm.reset()
        arm.home()
        arm.set_joint_speed(20)  # deg/s: slow for the first tests
else:
    arm = SimulatedRobot()


def guard():
    """Real arm: if the block is interrupted or fails, the arm goes back to its rest position."""
    return arm.guard() if LIVE_ROBOT else contextlib.nullcontext()


def grab(camera_id=CAMERA_ID, size=(1280, 720)):
    """One frame of the USB camera (a few frames are skipped to let the exposure settle)."""
    cap = cv2.VideoCapture(camera_id, cv2.CAP_DSHOW)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, size[0]); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, size[1])
    for _ in range(10):
        ok, frame = cap.read()
    cap.release()
    assert ok, f"no image from camera {camera_id}"
    return frame


def leave_view(robot, z_table, clear=150.0):
    """Lift the tool vertically, then fold the arm so that it does not hide the table from the camera."""
    pose = list(robot.get_tool_pose()[:5])
    pose[2] = max(pose[2], z_table + clear)
    safe_move(robot, pose, z_table)
    robot.go_to_sleep_pos()

In [ ]:
# Step 3 -- teach the markers (live robot only)
if LIVE_ROBOT:
    taught_live = {}
    with guard():
        for i in MARKERS_LIVE:
            input(f"Move the tool tip onto the centre of marker {i}, then press Enter")
            taught_live[i] = arm.get_tool_pose()[:3]
            print(i, taught_live[i])
    xyz_live = np.array([taught_live[i] for i in MARKERS_LIVE], float)
    A_live = fit_table_to_robot(np.array([MARKERS_LIVE[i] for i in MARKERS_LIVE]), xyz_live[:, :2])
    z_table_live = xyz_live[:, 2].mean()
    with guard():
        leave_view(arm, z_table_live)  # the tool is still touching the last marker
else:
    A_live, z_table_live = A_tr, z_table

In [ ]:
# Step 4 -- picture -> red circle -> robot frame -> move
if LIVE_CAMERA:
    fs = cv2.FileStorage("cell_calibration.yaml", cv2.FILE_STORAGE_READ)
    K_cell, dist_cell = fs.getNode("K").mat(), fs.getNode("dist").mat()
    fs.release()
    img, markers = grab(), MARKERS_LIVE
else:
    K_cell, dist_cell = K, dist
    img, markers = cv2.imread(str(DATA / "robot" / "robot_cell_2.png")), MARKERS_MM

uv, circ = find_red_circle(img)
assert uv is not None, "no red circle found: check the lighting and the mask (Lab 1)"
target = pixel_to_robot(uv, table_homography(img, K_cell, dist_cell, markers), A_live, K_cell, dist_cell)[0]
plt.figure(figsize=(9, 5)); plt.imshow(img[:, :, ::-1]); plt.plot(*uv, "c+", ms=25, mew=3); plt.axis("off")
plt.title(f"red circle at {uv.round(0)} px -> robot ({target[0]:.1f}, {target[1]:.1f}) mm")
print("target in the robot frame:", target.round(1))
with guard():
    go_above(arm, target, z_table_live)
    if LIVE_ROBOT:
        input("Check where the tool points, then press Enter: the arm folds away for the next picture")
    leave_view(arm, z_table_live)

**Going further.** Move the circle and run the last cell again; measure the error with a ruler (where does the tool tip point?). Put the circle near the edge of the image: is the error larger without the calibration of the cell camera? Tomorrow, the hand-made red-circle detector will be replaced by a neural network.